# Import

In [11]:
import shutil
from pathlib import Path
from dotenv import load_dotenv

import kagglehub
import hiddenlayer as hl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

load_dotenv()
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Using {DEVICE} for inference')

Using cpu for inference


In [2]:
from lab3.utils import get_dataloaders

# Data

In [3]:
data_dir = '/home/redduck/VSProjects/ITMO_SECS_CV_2025/lab3/data/input/PetImages'
train_loader, val_loader, test_loader = get_dataloaders(data_dir)

Downlod dataset to path: /home/redduck/.cache/kagglehub/datasets/bhavikjikadara/dog-and-cat-classification-dataset/versions/1/PetImages
An unexpected error occurred: Destination path '/home/redduck/VSProjects/ITMO_SECS_CV_2025/lab3/data/input/PetImages' already exists


# Model

## Classes & funcs

In [12]:
class CustomResNet50(nn.Module):
    def __init__(self, num_classes=2, freeze_backbone=True):
        super().__init__()

        self.backbone = models.resnet50(pretrained=True)

        # Кастомный avgpool
        self.backbone.avgpool = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        in_features = self.backbone.fc.in_features

        # Кастомный классификатор
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

        # Заморозка backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

            # Размораживаем только fc
            for param in self.backbone.fc.parameters():
                param.requires_grad = True

    def forward(self, x):
        return self.backbone(x)
    

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        running_loss += loss.item() * labels.size(0)

    return running_loss / total, correct / total


def train_model(
        model,
        train_loader, val_loader,
        epochs,
        criterion,
        optimizer,
        device='cpu'):
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    return model


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        running_loss += loss.item() * labels.size(0)

    return running_loss / total, correct / total

In [5]:
# # https://github.com/waleedka/hiddenlayer/blob/master/demos/pytorch_graph.ipynb
# transforms = [hl.transforms.Prune('Constant')] # Removes Constant nodes from graph.

# # Rather than using the default transforms, build custom ones to group
# # nodes of residual and bottleneck blocks.
# transforms = [
#     # Fold Conv, BN, RELU layers into one
#     hl.transforms.Fold("Conv > BatchNorm > Relu", "ConvBnRelu"),
#     # Fold Conv, BN layers together
#     hl.transforms.Fold("Conv > BatchNorm", "ConvBn"),
#     # Fold bottleneck blocks
#     hl.transforms.Fold("""
#         ((ConvBnRelu > ConvBnRelu > ConvBn) | ConvBn) > Add > Relu
#         """, "BottleneckBlock", "Bottleneck Block"),
#     # Fold residual blocks
#     hl.transforms.Fold("""ConvBnRelu > ConvBnRelu > ConvBn > Add > Relu""",
#                        "ResBlock", "Residual Block"),
#     # Fold repeated blocks
#     hl.transforms.FoldDuplicates(),
# ]

# # Display graph using the transforms above
# graph = hl.build_graph(resnet50, torch.zeros([1, 3, 224, 224]), transforms=transforms)
# graph.theme = hl.graph.THEMES['blue'].copy()
# graph.save('rnn_hiddenlayer', format='png')

## Train

In [13]:
model = CustomResNet50()

# Parameters
epochs = 5
lr = 1e-3

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/redduck/VSProjects/ITMO_SECS_CV_2025/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [14]:
train_model(
    model,
    train_loader, val_loader,
    epochs,
    criterion,
    optimizer,
    DEVICE
)

  0%|          | 0/547 [00:00<?, ?it/s]

  1%|          | 5/547 [00:27<49:20,  5.46s/it]


KeyboardInterrupt: 

## Evaluate

In [ ]:
evaluate(model, test_loader, criterion, DEVICE)